<a href="https://colab.research.google.com/github/muqaddaszaheer/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muqaddaszaheer/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

The research question is: can observable content and performance signals be used to identify pages that may show declining performance and help prioritize refresh review?

The decision supported is which pages should be reviewed first by an editor or content team. The output is a ranked review priority, not an automatic publishing or ranking decision.

The analysis uses anonymized content-level observations and is intended for directional decision support.

In [4]:
print("Section 1: Research question")
print("Decision: prioritize pages for refresh review.")
print("Output: ranked review priority.")
print("Use: directional decision support.")

Section 1: Research question
Decision: prioritize pages for refresh review.
Output: ranked review priority.
Use: directional decision support.


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

The analysis uses the bundled FlyRank anonymized content-refresh dataset with 30,000 rows and 44 source columns.

The target is `is_declining_label`, which represents the observed declining-performance label. Client and content identifiers are used only for grouping and validation checks and are not used as model features.

The model deliberately excludes target-derived and action-derived fields such as `trend_direction`, `trend_pct`, `is_declining`, `is_quick_win`, `needs_ctr_fix`, `needs_engagement_fix`, `ai_opportunity`, and `is_initial_refresh_candidate`.

No client names, domains, URLs, titles, keywords, or private queries are used in the analysis.

In [5]:
from pathlib import Path
import subprocess
import os

repo_url = "https://github.com/muqaddaszaheer/flyrank-ml-internship.git"
repo_dir = Path("/content/flyrank-ml-internship")

# Clone the repository if it is not already available
if not repo_dir.exists():
    subprocess.run(
        ["git", "clone", repo_url, str(repo_dir)],
        check=True
    )

# Move into the repository
os.chdir(repo_dir)

print("Repository ready.")
print("Location:", Path.cwd())
print("Files:", len(list(repo_dir.rglob("*"))))

Repository ready.
Location: /content/flyrank-ml-internship
Files: 133


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

The final analysis uses Logistic Regression as an interpretable reference model and the repository's model pipeline for model comparison.

The target is `is_declining_label`.

The model uses observable numeric and categorical content/performance features. Target-derived fields, action flags, and pseudonymous identifiers are excluded from the feature set.

The final validation uses a client-holdout split so pages from the same client are not placed in both training and test sets. Random seed 42 is used for reproducibility.

The leakage audit checks that the target and known target-derived or action-derived fields are not present in the final feature set.

In [6]:
import sys
import subprocess

scripts_dir = repo_dir / "scripts"

if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

from ml_utils import (
    MODEL_NUMERIC_FEATURES,
    MODEL_CATEGORICAL_FEATURES,
)

print("Numeric model features:")
for feature in MODEL_NUMERIC_FEATURES:
    print("-", feature)

print("\nCategorical model features:")
for feature in MODEL_CATEGORICAL_FEATURES:
    print("-", feature)

print("\nRandom seed: 42")
print("Validation design: client holdout")
print("Target: is_declining_label")

forbidden_features = {
    "is_declining_label",
    "trend_direction",
    "trend_pct",
    "is_declining",
    "is_quick_win",
    "needs_ctr_fix",
    "needs_engagement_fix",
    "ai_opportunity",
    "is_underperformer",
    "is_initial_refresh_candidate",
}

all_features = set(MODEL_NUMERIC_FEATURES) | set(MODEL_CATEGORICAL_FEATURES)
leakage_hits = sorted(all_features.intersection(forbidden_features))

print("\nLeakage intersections:", leakage_hits)

if leakage_hits:
    raise ValueError(f"Leakage detected: {leakage_hits}")

print("PASS: no forbidden leakage features found.")

Numeric model features:
- search_volume
- competition
- cpc
- word_count
- char_count
- log_impressions_90d
- log_clicks_90d
- log_sessions_90d
- log_ai_sessions_90d
- days_with_impressions
- days_with_sessions
- content_age_days
- days_since_last_update
- ctr
- avg_position
- engagement_rate
- scroll_rate
- ai_traffic_pct

Categorical model features:
- competition_level
- content_type
- main_intent
- age_tier
- freshness_tier
- word_count_tier
- impression_tier
- position_tier

Random seed: 42
Validation design: client holdout
Target: is_declining_label

Leakage intersections: []
PASS: no forbidden leakage features found.


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

The model comparison uses the same evaluation framework and a client-holdout validation design.

The repository's evaluated results show that the random forest performed best on Precision@50, with a measured Precision@50 of 0.740 compared with 0.240 for the baseline rules. This is approximately a 3.1x improvement in the number of declining pages found among the top 50 ranked items.

The random forest also achieved a measured ROC AUC of 0.750, average precision of 0.618, recall of 0.744, and F1 of 0.640 on the reported evaluation.

These results are measured on this dataset and should be interpreted as directional evidence rather than a guarantee of future performance.

In [7]:
import pandas as pd

results = pd.DataFrame({
    "Model": [
        "Baseline rules",
        "Logistic Regression",
        "Decision Tree",
        "Random Forest",
    ],
    "ROC AUC": [
        0.627,
        0.700,
        0.742,
        0.750,
    ],
    "Average Precision": [
        0.468,
        0.522,
        0.575,
        0.618,
    ],
    "Precision@50": [
        0.240,
        0.400,
        0.540,
        0.740,
    ],
    "Recall": [
        None,
        0.567,
        0.716,
        0.744,
    ],
    "F1": [
        None,
        0.566,
        0.634,
        0.640,
    ],
})

display(results)

lift = 0.740 / 0.240

print(f"Random Forest Precision@50 lift over baseline: {lift:.2f}x")
print("Section 4 completed successfully.")

,Model,ROC AUC,Average Precision,Precision@50,Recall,F1
0,Baseline rules,0.627,0.468,0.24,NaN,NaN
1,Logistic Regression,0.700,0.522,0.40,0.567,0.566
2,Decision Tree,0.742,0.575,0.54,0.716,0.634
3,Random Forest,0.750,0.618,0.74,0.744,0.640


Random Forest Precision@50 lift over baseline: 3.08x
Section 4 completed successfully.


## 5. Limitations

*What this work cannot claim.*

This analysis has several limitations.

First, the dataset is anonymized and represents the available FlyRank internship data, so the results may not generalize to every website or content environment.

Second, the model identifies directional signals associated with the observed declining label. It does not establish causation and does not predict Google's ranking algorithm.

Third, model errors remain on unseen clients. The ML-09 validation audit recorded misclassified test rows, showing that the model is not perfect.

Finally, recommendations should be reviewed by a human before any content change is made.

In [8]:
limitations = [
    "Results are specific to the available anonymized dataset.",
    "The model does not establish causal relationships.",
    "The model does not predict Google's ranking algorithm.",
    "The model makes errors on unseen clients.",
    "Recommendations require human review before action."
]

print("Limitations")
print("=" * 60)

for number, limitation in enumerate(limitations, start=1):
    print(f"{number}. {limitation}")

print("\nSection 5 completed successfully.")

Limitations
1. Results are specific to the available anonymized dataset.
2. The model does not establish causal relationships.
3. The model does not predict Google's ranking algorithm.
4. The model makes errors on unseen clients.
5. Recommendations require human review before action.

Section 5 completed successfully.


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

The ranked recommendations are intended to help an editor decide where to review first.

1. Review high-confidence refresh candidates first, especially when the model shows decline risk together with visible performance signals.
2. Review click-through performance when a page has sufficient impressions, reasonable position, and weak CTR.
3. Review engagement when there is enough session activity but weak engagement or scroll signals.
4. Expand and refresh thin visible pages when the available evidence supports both content improvement and refresh.
5. Monitor pages where the available signals do not provide enough evidence for a confident action.

The safest use is to inspect high-priority items manually, verify the evidence, and then decide whether an editorial action is appropriate.

In [9]:
recommendations = pd.DataFrame([
    {
        "Priority": 1,
        "Action": "Review high-confidence refresh candidates",
        "Reason": "Model decline risk plus supporting performance signals"
    },
    {
        "Priority": 2,
        "Action": "Review click-through performance",
        "Reason": "Sufficient impressions with weak CTR"
    },
    {
        "Priority": 3,
        "Action": "Review engagement",
        "Reason": "Enough sessions with weak engagement or scroll signals"
    },
    {
        "Priority": 4,
        "Action": "Expand and refresh thin pages",
        "Reason": "Thin visible content with supporting evidence"
    },
    {
        "Priority": 5,
        "Action": "Monitor",
        "Reason": "Insufficient evidence for a confident intervention"
    },
])

display(recommendations)

print("Section 6 completed successfully.")

,Priority,Action,Reason
0,1,Review high-confidence refresh candidates,Model decline risk plus supporting performance...
1,2,Review click-through performance,Sufficient impressions with weak CTR
2,3,Review engagement,Enough sessions with weak engagement or scroll...
3,4,Expand and refresh thin pages,Thin visible content with supporting evidence
4,5,Monitor,Insufficient evidence for a confident interven...


Section 6 completed successfully.


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

The paper should embed the model-versus-baseline comparison table and the main visual artifacts produced by the repository pipeline.

The repository pipeline generates a ranked refresh queue, model results, a summary, and charts covering action mix, confidence mix, reason codes, feature importance, and trend distribution.

These artifacts make the analysis easier to inspect and reproduce.

In [11]:
from pathlib import Path
import subprocess
import sys
import os

repo_dir = Path("/content/flyrank-ml-internship")
outputs_dir = repo_dir / "outputs"

# Make sure we are in the repository
if not repo_dir.exists():
    raise FileNotFoundError(
        f"Repository not found: {repo_dir}"
    )

os.chdir(repo_dir)

# Install the repository requirements
print("Installing required packages...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    check=True
)

# Run the official FlyRank pipeline and show the real error if it fails
print("\nRunning the official pipeline...\n")

result = subprocess.run(
    [sys.executable, "scripts/run_all.py"],
    cwd=repo_dir,
    text=True,
    capture_output=True
)

print(result.stdout)

if result.returncode != 0:
    print("\n--- PIPELINE ERROR ---\n")
    print(result.stderr)
    raise RuntimeError(
        f"Pipeline failed with exit code {result.returncode}. "
        "The detailed error is shown above."
    )

print("\nPipeline completed successfully!")

print("\nGenerated artifacts:")
for path in sorted(outputs_dir.rglob("*")):
    if path.is_file():
        print("-", path.relative_to(repo_dir))

print("\nSection 7 completed successfully.")

Installing required packages...

Running the official pipeline...


▶ Step 1/5 — Prepare features — clean the data, build the feature vector, define the label
Prepared 30,000 rows from 30,000 raw rows
Wrote /content/flyrank-ml-internship/data/processed/refresh_feature_vector.csv

▶ Step 2/5 — Baseline — a transparent hand-written rule to beat
Wrote baseline queue: /content/flyrank-ml-internship/data/processed/baseline_refresh_queue.csv
Top-50 declining rate (full data, not the evaluated holdout Precision@50): 0.340

▶ Step 3/5 — Train — logistic regression, decision tree, random forest (client-holdout split)
Trained 3 models on 30,000 rows
Split strategy: client_holdout
Best model: random_forest
Wrote predictions: /content/flyrank-ml-internship/data/processed/model_predictions.csv
Wrote model results: /content/flyrank-ml-internship/outputs/model_results.json

▶ Step 4/5 — Evaluate — ranked refresh queue, charts, and the Markdown report
Wrote final refresh queue: /content/flyrank-ml-int

## ML-12 — Demo, social post, and employer summary

### 5-minute demo outline

**0:00–0:45 — Problem**
Explain that the project helps prioritize pages for refresh review using observable content and performance signals.

**0:45–1:30 — Data and safety**
Show the anonymized dataset and explain that client-identifying information, URLs, titles, keywords, and target-derived leakage fields are excluded from model features.

**1:30–2:30 — Method**
Explain the baseline, model comparison, and client-holdout validation. Show that train and test clients are separated.

**2:30–3:30 — Results**
Show the model comparison and explain that Random Forest achieved Precision@50 of 0.740 compared with 0.240 for the baseline rules in the reported evaluation.

**3:30–4:15 — Action output**
Show the ranked refresh queue and explain how an editor can review high-priority items.

**4:15–5:00 — Limitation and guardrail**
Explain that the model is directional decision support, not a guarantee of future performance or a prediction of Google's ranking algorithm. Human review is required before taking action.

### Social-post cut

I completed a machine-learning capstone focused on prioritizing content for refresh review. I compared transparent baseline rules with machine-learning models, used client-holdout validation, checked for leakage, and turned the results into ranked editorial recommendations. The main lesson was that strong ML work is not only about the model — it also requires careful validation, honest claims, reproducibility, and a clear path from analysis to action.

### Employer-facing summary

I built an end-to-end machine-learning workflow for content refresh prioritization using anonymized data. I compared baseline rules with multiple models, performed client-holdout validation, audited the feature set for leakage, and produced a ranked action queue. The project demonstrates practical skills in Python, pandas, scikit-learn, model evaluation, data safety, and decision-support reporting.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
